<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Novelty and learning effects

**[Novelty and learning effects](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/)** · Experiments and Causal Impact: Did Our Change Work? · Module 4, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see a novelty effect in a simulated test: the effect by calendar week, the effect by week since each customer first saw the change, and why the second one shows the long-run effect.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | the result and its interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); peeking, from [Peeking](https://learning.nextia-ai.com/courses/experiments/m04/peeking/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M04-L03-novelty-and-learning-effects/novelty-and-learning-effects-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A widget that is new, then old

A new widget helps a customer in the first days after they first see it, then hurts a little. The simulator in the project uses this curve for the chance to order:

chance with the widget = chance without it × (1 + 0.12 × e^(−days / 5) − 0.03)

`days` is the number of days since the customer first saw the widget.

> **Predict.** In which week since the first visit does the effect become negative?

In [ ]:
def lift(days):
    return 1 + 0.12 * np.exp(-days / 5) - 0.03

for d in [0, 3, 7, 10, 14, 21, 28]:
    print(f"day {d:>2}: {lift(d) - 1:+.1%}")

> **Check.** You should see `day  0: +9.0%`, `day  7: -0.0%` and `day 28: -3.0%`.

The effect starts at +9%, crosses zero after about a week, and settles at −3%. The long-run effect is negative.

## 2. Simulate a 6-week test

30,000 customers. Each one first visits on some day (most in the first weeks), then comes back on about 15% of the days. We simulate both worlds with the same random numbers, so we know the truth: `order0` without the widget, `order1` with it.

In [ ]:
def novelty_test(n=30000, days=42, p0=0.06, basket=50.0, seed=0):
    rng = np.random.default_rng(seed)
    first = np.minimum(rng.geometric(0.08, n) - 1, 99)       # the day of each customer's first visit
    b = rng.random(n) < 0.5                                   # the coin flip per customer
    since = np.arange(days)[None, :] - first[:, None]          # days since the first visit
    visit = (since == 0) | ((since > 0) & (rng.random((n, days)) < 0.15))
    u = rng.random((n, days))
    order0 = visit & (u < p0)                                 # the world without the widget
    order1 = visit & (u < p0 * lift(np.maximum(since, 0)))    # the world with it (same random numbers)
    rows, cols = np.nonzero(visit)
    return pd.DataFrame({
        "customer": rows, "variant": np.where(b[rows], "B", "A"),
        "week": cols // 7 + 1, "week_since_first": since[rows, cols] // 7 + 1,
        "revenue": np.where(b[rows], order1[rows, cols], order0[rows, cols]) * basket,
        "rev0": order0[rows, cols] * basket, "rev1": order1[rows, cols] * basket})

v = novelty_test()
a, b = v[v.variant == "A"].revenue, v[v.variant == "B"].revenue
print(f"{len(v):,} visits; revenue per visit A {a.mean():.3f}, B {b.mean():.3f}: {b.mean() / a.mean() - 1:+.1%}")
print(f"true effect over the 6 weeks: {v.rev1.mean() / v.rev0.mean() - 1:+.1%}")

> **Check.** You should see `163,774 visits`, an estimate of `+1.3%` and `true effect over the 6 weeks: +0.8%`.

The six-week average is small and positive. It hides two different stories.

## 3. By calendar week

In [ ]:
def by(col):
    out = {}
    for w, part in v.groupby(col):
        a, b = part[part.variant == "A"].revenue, part[part.variant == "B"].revenue
        out[w] = {"estimate": b.mean() / a.mean() - 1, "truth": part.rev1.mean() / part.rev0.mean() - 1,
                  "first_week_share": (part.week_since_first == 1).mean(), "visits": len(part)}
    return pd.DataFrame(out).T.round(3)

by("week")

> **Check.** In week 1, `estimate` is about 0.099 and `first_week_share` is 1.0: every visit in week 1 is a first-week visit. By week 6, the share is about 0.05.

A team that stops after two weeks sees about +10% and ships a widget that loses money later.

## 4. Your turn: by week since the first visit

> **Your turn.** Compute the same table by `week_since_first`. Put in `long_run` the mean of the `truth` column for weeks 3 to 6 since the first visit, rounded to 3 decimals.

In [ ]:
table = by("week_since_first")
long_run = round(float(table.loc[3:6, "truth"].mean()), 3)
print(table)
print(long_run)

In [ ]:
expect('long_run', long_run, -0.026, tolerance=0.0005)

By week since the first visit, the true effect is positive only in week 1. Weeks 3 to 6 show the long run: about −2.6%. The estimates are noisier than the truth, so read their direction over several weeks, not one number.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/) explains novelty and learning effects with Larkfield's widget, and has the knowledge checks that count toward your certificate. In the project, `sim run --scenario novelty` and `xp weekly` give both tables, and `xp check` compares them with the truth.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [When units affect each other](https://learning.nextia-ai.com/courses/experiments/m04/when-units-affect-each-other/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.